# Notebook 1: Dataset Collection and Text Preprocessing

## Objective
This notebook performs dataset loading, exploration, and preprocessing for the Fake News Detection NLP project.

## Dataset
Fake and Real News Dataset from Kaggle.

## Member
Pesara (CIT-24-01-0258)

## Models Assigned
Machine Learning Model: SVM
Deep Learning Model: LSTM

# Import Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import string

import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download NLTK Resources

In [ ]:
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Load Dataset

In [3]:
fake = pd.read_csv("../data/Fake.csv")
real = pd.read_csv("../data/True.csv")

In [4]:
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [5]:
real.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


# Check Dataset Shape

In [6]:
print("Fake News Dataset Shape:", fake.shape)

print("Real News Dataset Shape:", real.shape)

Fake News Dataset Shape: (23481, 4)
Real News Dataset Shape: (21417, 4)


# Check Columns

In [7]:
print(fake.columns)

print(real.columns)

Index(['title', 'text', 'subject', 'date'], dtype='str')
Index(['title', 'text', 'subject', 'date'], dtype='str')


# Add Labels

Machine Learning needs labels.

Fake = 0

Real = 1

In [8]:
fake["label"] = 0

real["label"] = 1

# Merge Both Datasets then call it ''News''

In [9]:
news = pd.concat([fake, real], ignore_index=True)

# Check New Dataset

In [10]:
news.head()

,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0


In [11]:
news.info()

<class 'pandas.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   title    44898 non-null  str  
 1   text     44898 non-null  str  
 2   subject  44898 non-null  str  
 3   date     44898 non-null  str  
 4   label    44898 non-null  int64
dtypes: int64(1), str(4)
memory usage: 112.3 MB


# Check Missing Values

In [12]:
news.isnull().sum()

title      0
text       0
subject    0
date       0
label      0
dtype: int64

# Check Duplicate Records and remove them

In [13]:
news.duplicated().sum()

np.int64(209)

In [14]:
news = news.drop_duplicates()

# Shuffle Dataset

In [15]:
news = news.sample(frac=1, random_state=42).reset_index(drop=True)

# Keep Only Required Columns

In [16]:
news = news[["title","text","label"]]

In [17]:
news.head()

,title,text,label
0,WOW! LEFTIST LIBRARIAN REJECTS Shipment Of Chi...,"A school librarian in Cambridge, Massachusetts...",0
1,Kenya opposition leader calls for calm in slum...,NAIROBI (Reuters) - Kenyan opposition leader R...,1
2,Egypt rejects U.S. decision to move its embass...,CAIRO (Reuters) - Egypt rejected the U.S. deci...,1
3,(AUDIO)NATION OF ISLAM LEADER FARRAKHAN: “WE W...,After a recent speech given by Minister Louis ...,0
4,Trump Rally Nearly Turns Into A Full-Blown Ra...,Tensions ran high outside of a campaign rally ...,0


# Combine Title + Text

improves performance

In [18]:
news["content"] = news["title"] + " " + news["text"]

In [19]:
news[["content","label"]].head()

,content,label
0,WOW! LEFTIST LIBRARIAN REJECTS Shipment Of Chi...,0
1,Kenya opposition leader calls for calm in slum...,1
2,Egypt rejects U.S. decision to move its embass...,1
3,(AUDIO)NATION OF ISLAM LEADER FARRAKHAN: “WE W...,0
4,Trump Rally Nearly Turns Into A Full-Blown Ra...,0


# Create Stopwords & Lemmatizer

In [20]:
stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()

# Create Cleaning Function

In [21]:
def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", "", text)

    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

# Apply Cleaning

In [22]:
news["content"] = news["content"].apply(clean_text)

# Tokenization

In [23]:
nltk.download("punkt_tab")
news["tokens"] = news["content"].apply(word_tokenize)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# Remove Stopwords

In [24]:
news["tokens"] = news["tokens"].apply(

    lambda words: [

        word

        for word in words

        if word not in stop_words

    ]

)

# Lemmatization

In [25]:
news["tokens"] = news["tokens"].apply(

    lambda words: [

        lemmatizer.lemmatize(word)

        for word in words

    ]

)

# Convert Tokens Back to Sentence

In [26]:
news["clean_text"] = news["tokens"].apply(

    lambda words: " ".join(words)

)

# Final Dataset

In [27]:
news[["clean_text","label"]].head()

,clean_text,label
0,wow leftist librarian reject shipment child ’ ...,0
1,kenya opposition leader call calm slum hit dea...,1
2,egypt reject u decision move embassy jerusalem...,1
3,audionation islam leader farrakhan “ kill ” go...,0
4,trump rally nearly turn fullblown race war st ...,0


# Save Clean Dataset

In [28]:
import os

os.makedirs("data/processed", exist_ok=True)

news.to_csv("../data/processed/clean_news.csv", index=False)

print("Dataset saved successfully!")

Dataset saved successfully!


In [29]:
print(news.shape)

news.head()

(44689, 6)


,title,text,label,content,tokens,clean_text
0,WOW! LEFTIST LIBRARIAN REJECTS Shipment Of Chi...,"A school librarian in Cambridge, Massachusetts...",0,wow leftist librarian rejects shipment of chil...,"[wow, leftist, librarian, reject, shipment, ch...",wow leftist librarian reject shipment child ’ ...
1,Kenya opposition leader calls for calm in slum...,NAIROBI (Reuters) - Kenyan opposition leader R...,1,kenya opposition leader calls for calm in slum...,"[kenya, opposition, leader, call, calm, slum, ...",kenya opposition leader call calm slum hit dea...
2,Egypt rejects U.S. decision to move its embass...,CAIRO (Reuters) - Egypt rejected the U.S. deci...,1,egypt rejects us decision to move its embassy ...,"[egypt, reject, u, decision, move, embassy, je...",egypt reject u decision move embassy jerusalem...
3,(AUDIO)NATION OF ISLAM LEADER FARRAKHAN: “WE W...,After a recent speech given by Minister Louis ...,0,audionation of islam leader farrakhan “we will...,"[audionation, islam, leader, farrakhan, “, kil...",audionation islam leader farrakhan “ kill ” go...
4,Trump Rally Nearly Turns Into A Full-Blown Ra...,Tensions ran high outside of a campaign rally ...,0,trump rally nearly turns into a fullblown race...,"[trump, rally, nearly, turn, fullblown, race, ...",trump rally nearly turn fullblown race war st ...
